# 2. Feature Engineering\nDetailed feature preparation notebook aligned with the production pipeline.

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config.core import config\nfrom src.pipeline import pipe\nfrom src.processing.data_manager import load_dataset\nfrom src.processing.features import Mapper\n\nROOT = Path.cwd()\nif not (ROOT / 'src').exists():\n    ROOT = ROOT.parent

In [ ]:
data = load_dataset(config.app_config.training_data_file)\nprint('Shape after dropping configured columns:', data.shape)\ndisplay(data.head(3))

In [ ]:
x = data.drop(columns=[config.ml_config.target]).copy()\ny = data[config.ml_config.target].astype(int).copy()\n\nx_train, x_valid, y_train, y_valid = train_test_split(\n    x,\n    y,\n    test_size=config.ml_config.test_size,\n    random_state=config.ml_config.random_state,\n    stratify=y,\n)\n\nprint('x_train:', x_train.shape, '| x_valid:', x_valid.shape)

In [ ]:
x_manual = x_train.copy()\n\nfor col in config.ml_config.categorical_missing_with_constant:\n    x_manual[col] = x_manual[col].fillna('Missing')\n\nfor col in config.ml_config.categorical_missing_with_frequent:\n    mode_val = x_manual[col].mode(dropna=True)[0]\n    x_manual[col] = x_manual[col].fillna(mode_val)\n\nmissing_after_impute = x_manual.isna().sum().sum()\nprint('Total missing values after manual categorical imputation:', missing_after_impute)

In [ ]:
experience_mapper = Mapper(\n    variables=config.ml_config.experience_features,\n    mappings=config.ml_config.experience_map,\n    default_value=0,\n)\nlast_job_mapper = Mapper(\n    variables=config.ml_config.last_new_job_features,\n    mappings=config.ml_config.last_new_job_map,\n    default_value=0,\n)\ncompany_size_mapper = Mapper(\n    variables=config.ml_config.company_size_features,\n    mappings=config.ml_config.company_size_map,\n    default_value=0,\n)\n\nx_mapped = experience_mapper.transform(x_manual)\nx_mapped = last_job_mapper.transform(x_mapped)\nx_mapped = company_size_mapper.transform(x_mapped)\n\ndisplay(x_mapped[config.ml_config.experience_features + config.ml_config.last_new_job_features + config.ml_config.company_size_features].head())

In [ ]:
prep_pipe = pipe[:-1]\nx_train_transformed = prep_pipe.fit_transform(x_train, y_train)\nx_valid_transformed = prep_pipe.transform(x_valid)\n\nprint('Transformed train shape:', x_train_transformed.shape)\nprint('Transformed valid shape:', x_valid_transformed.shape)\nprint('NaN count in transformed train:', np.isnan(x_train_transformed).sum())\nprint('NaN count in transformed valid:', np.isnan(x_valid_transformed).sum())

In [ ]:
full_pipe = pipe\nfull_pipe.fit(x_train, y_train)\ntrain_score = full_pipe.score(x_train, y_train)\nvalid_score = full_pipe.score(x_valid, y_valid)\n\nprint(f'Train accuracy: {train_score:.4f}')\nprint(f'Valid accuracy: {valid_score:.4f}')

This notebook demonstrates feature processing both manually (for explainability) and through the final sklearn pipeline (for production consistency).